In [ ]:
!pip install transformers datasets sentencepiece sacrebleu accelerate

In [ ]:
!pip install -U transformers accelerate

In [ ]:
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
dataset_smiti = load_dataset("opus100", "en-sv")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
dataset_smiti["train"][0]

{'translation': {'en': 'The Icelandic authorities submitted comments on this Decision by letter dated 24 February 2005 (Event No 311243).',
  'sv': 'De isländska myndigheterna kommenterade beslutet i en skrivelse av den 24 februari 2005 (diarienummer 311243).'}}

In [ ]:
model_name_smiti = "Helsinki-NLP/opus-mt-en-sw"

tokenizer_smiti = MarianTokenizer.from_pretrained(model_name_smiti)
model_smiti = MarianMTModel.from_pretrained(model_name_smiti)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
def preprocess_function_smiti(examples):
  inputs_smiti = [ex["en"] for ex in examples["translation"]]
  targets_smiti = [ex["sv"] for ex in examples["translation"]]

  model_inputs_smiti = tokenizer_smiti(inputs_smiti, text_target=targets_smiti, max_length=128, truncation=True, padding="max_length")
  return model_inputs_smiti

In [ ]:
tokenized_datasets_smiti = dataset_smiti.map(preprocess_function_smiti, batched=True, remove_columns=dataset_smiti["train"].column_names)
small_train_smiti = tokenized_datasets_smiti["train"].shuffle(seed=42).select(range(1000))
small_valid_smiti = tokenized_datasets_smiti["validation"].shuffle(seed=42).select(range(200))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args_smiti = Seq2SeqTrainingArguments(
    output_dir="./mt-low-resource",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=1,
    logging_steps =100,
    report_to = "none"
)

In [ ]:
trainer_smiti = Seq2SeqTrainer(
    model=model_smiti,
    args=training_args_smiti,
    train_dataset=small_train_smiti,
    eval_dataset=small_valid_smiti,
    tokenizer=tokenizer_smiti,
)

/tmp/ipython-input-2278208089.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_smiti = Seq2SeqTrainer(


In [ ]:
trainer_smiti.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,2.149500


Step,Training Loss
100,2.149500


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[58949]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=125, training_loss=1.9639185485839843, metrics={'train_runtime': 1115.7653, 'train_samples_per_second': 0.896, 'train_steps_per_second': 0.112, 'total_flos': 33898364928000.0, 'train_loss': 1.9639185485839843, 'epoch': 1.0})

In [ ]:
text_smiti = "Machine learning is changing the world"
inputs_smiti = tokenizer_smiti(text_smiti, return_tensors="pt", padding=True)
translated = model_smiti.generate(**inputs_smiti)

print("vaishu_Translation:", tokenizer_smiti.decode(translated[0], skip_special_tokens=True))

vaishu_Translation: Mafunzo ya mashine yanabadili ulimwengu
